# Structured Output 
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easilt parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027A1DCD9040>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027A1DFDC410>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="Ratings on the movie out of 10")
    

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027A1DCD9040>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027A1DFDC410>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'Ratings on the movie out of 10', 'type': 'number'}}, 'required': ['title', 'year',

In [4]:
model.invoke("Provide the details about the movie Inception")

AIMessage(content='<think>\nOkay, the user is asking for details about the movie Inception. Let me start by recalling what I know. Inception is a 2010 sci-fi action film directed by Christopher Nolan. It stars Leonardo DiCaprio as Dom Cobb, a thief who enters people\'s dreams to steal secrets. The main plot revolves around the concept of planting an idea into someone\'s mind, which is called "inception." \n\nFirst, I need to outline the basic information: director, writer, main cast, release date, genre, and runtime. Then, the plot summary. The user might want to know the main characters and their roles. Dom Cobb is the protagonist, and the target is Robert Fischer, whose father needs to be influenced. The team includes Arthur, Eames, Ariadne, and Yusuf. \n\nThe structure of the movie is a bit complex with layers of dreams, so explaining the three levels (real world, level 1, level 2, level 3) would be important. Each level has different time dilation, which affects the story. The conc

In [5]:
model_with_structure.invoke("Provide the details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structured

In [6]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="Ratings on the movie out of 10")

model_with_structure=model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There's a Movie function that requires title, year, director, and rating. I need to fill those parameters. I know Inception was directed by Christopher Nolan. It came out in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the exact year and rating. Yep, 2010 and 8.8. So I'll structure the tool call with those details.\n", 'tool_calls': [{'id': 't46sth4z4', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 159, 'prompt_tokens': 227, 'total_tokens': 386, 'completion_time': 0.279228059, 'completion_tokens_details': {'reasoning_tokens': 111}, 'prompt_time': 0.011466644, 'prompt_tokens_details': None, 'queue_time': 0.232805461, 'total_time': 0.290694

### Nested Structure

In [7]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget is million USD")

In [8]:
model_with_structure=model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

## TypedDict
TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [10]:
from typing_extensions import TypedDict, Annotated

In [11]:
class movieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie."]
    year: Annotated[int, ..., "The year movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [13]:
model_withtypeddict=model.with_structured_output(movieDict)
response=model_withtypeddict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [15]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, description="Budget is million USD")

model_with_structure = model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Please provide the details of the movie avengers")
response


{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers',
 'year': 2012}

In [16]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

# Dataclasses
A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator.

In [17]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [20]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information fo a person"""
    name: str=Field(description="The name address of the person")
    email:str=Field(description="The email address of the person")
    phone:str=Field(description="The phone number of a person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from  John Doe, john@example.com, (555) 123-4567"}]
})
result

{'messages': [HumanMessage(content='Extract contact info from  John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='b460b391-d00f-42f5-ad18-82da827b8c43'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 804, 'prompt_tokens': 205, 'total_tokens': 1009, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DN3KlGKGhW2bx8uArdKjRRX3vshaF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d21b0-d7cf-7f13-b63f-28c02d037f6f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'outpu

In [21]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [22]:
from typing_extensions import TypedDict, Annotated

class ContactInfo(TypedDict):
    """Contact information fo a person"""
    name: str
    email:str
    phone:str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from  John Doe, john@example.com, (555) 123-4567"}]
})
result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [26]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information fo a person"""
    name: str
    email:str
    phone:str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from  John Doe, john@example.com, (555) 123-4567"}]
})
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')